In [2]:
# =============================================================================
# TASK 1: DATA INVENTORY & UNDERSTANDING (COMPLETE)
# =============================================================================
# What we're doing: Load all 4 AFL data files, check quality, fix issues
# Why: Before building ANY model, we must understand what data we have
# =============================================================================

import pandas as pd
import numpy as np
import os

# --- STEP 1: Load All Files ---
data_path = r"C:\Internship\Netixsol\week-3\day-1\data\afl_datasets"

players_info = pd.read_csv(os.path.join(data_path, "afl_players_info_raw.csv"))
round_stats = pd.read_csv(os.path.join(data_path, "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"))
seasonal_stats = pd.read_csv(os.path.join(data_path, "afl_players_seasonal_stats_raw.csv"), low_memory=False)
matches = pd.read_csv(os.path.join(data_path, "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv"))

print("All 4 files loaded successfully!")
print(f"Players Info:    {players_info.shape[0]:>8,} rows x {players_info.shape[1]} columns")
print(f"Round Stats:     {round_stats.shape[0]:>8,} rows x {round_stats.shape[1]} columns")
print(f"Seasonal Stats:  {seasonal_stats.shape[0]:>8,} rows x {seasonal_stats.shape[1]} columns")
print(f"Matches:         {matches.shape[0]:>8,} rows x {matches.shape[1]} columns")

# --- STEP 2: What Each File Contains ---
print("\n" + "=" * 70)
print("FILE 1: PLAYER INFO")
print("=" * 70)
print(f"Grain: 1 row = 1 player")
print(f"Columns: {list(players_info.columns)}")
display(players_info[['id', 'player_name', 'born_date', 'debut_date', 'height', 'weight', 'player_teams']].head(3))

print("\n" + "=" * 70)
print("FILE 2: ROUND-BY-ROUND STATS (Biggest file)")
print("=" * 70)
print(f"Grain: 1 row = 1 player in 1 game")
print(f"Columns: {list(round_stats.columns)}")
display(round_stats[['player_id', 'team', 'year', 'round', 'opponent', 'result', 'kicks', 'goals', 'disposals', 'fantasy_points']].head(3))

print("\n" + "=" * 70)
print("FILE 3: SEASONAL STATS")
print("=" * 70)
print(f"Grain: 1 row = 1 player in 1 season")
print(f"Columns: {list(seasonal_stats.columns)}")
display(seasonal_stats[['player_id', 'year', 'team', 'games_played', 'avg_kicks', 'avg_goals', 'avg_disposals']].head(3))

print("\n" + "=" * 70)
print("FILE 4: MATCH RESULTS")
print("=" * 70)
print(f"Grain: 1 row = 1 team in 1 match")
print(f"Columns: {list(matches.columns)}")
display(matches[['team_name', 'year', 'round', 'match_date', 'home_away', 'opponent', 'team_score', 'opponent_score', 'result', 'margin', 'venue']].head(3))

# --- STEP 3: Data Quality - Missing Values ---
print("\n" + "=" * 70)
print("DATA QUALITY CHECK: MISSING VALUES")
print("=" * 70)
print("(NaN = Not a Number = empty cell = missing data)\n")

for name, df in [("Players Info", players_info), 
                  ("Round Stats", round_stats), 
                  ("Seasonal Stats", seasonal_stats), 
                  ("Matches", matches)]:
    total_missing = df.isnull().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    pct = (total_missing / total_cells * 100) if total_cells > 0 else 0
    
    print(f"--- {name} ({df.shape[0]:,} rows) ---")
    if total_missing == 0:
        print(f"  ✅ No missing values!")
    else:
        print(f"  ⚠️  {total_missing:,} missing values ({pct:.1f}% of all cells)")
        missing_by_col = df.isnull().sum()
        missing_cols = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
        for col, count in missing_cols.items():
            col_pct = count / len(df) * 100
            print(f"     - {col}: {count:,} missing ({col_pct:.1f}%)")
    print()

# --- STEP 4: Data Quality - Duplicates ---
print("=" * 70)
print("DATA QUALITY CHECK: DUPLICATES")
print("=" * 70)
print("(Exact same row appearing more than once)\n")

for name, df in [("Players Info", players_info), 
                  ("Round Stats", round_stats), 
                  ("Seasonal Stats", seasonal_stats), 
                  ("Matches", matches)]:
    dupes = df.duplicated().sum()
    if dupes == 0:
        print(f"  ✅ {name}: No duplicates")
    else:
        print(f"  ⚠️  {name}: {dupes:,} duplicate rows!")
print()

# --- STEP 5: Data Coverage ---
print("=" * 70)
print("DATA COVERAGE")
print("=" * 70)

print(f"\n--- Years Covered ---")
print(f"  Round Stats:     {round_stats['year'].min()} to {round_stats['year'].max()} ({round_stats['year'].nunique()} unique years)")
print(f"  Seasonal Stats:  {seasonal_stats['year'].min()} to {seasonal_stats['year'].max()} ({seasonal_stats['year'].nunique()} unique years)")
print(f"  Matches:         {matches['year'].min()} to {matches['year'].max()} ({matches['year'].nunique()} unique years)")

print(f"\n--- Teams ---")
teams_in_matches = sorted(matches['team_name'].unique())
print(f"  Total teams: {len(teams_in_matches)}")
for i, team in enumerate(teams_in_matches, 1):
    print(f"    {i:2d}. {team}")

print(f"\n--- Players ---")
print(f"  Unique players in Player Info: {players_info['id'].nunique():,}")
print(f"  Unique players in Round Stats: {round_stats['player_id'].nunique():,}")

print(f"\n--- Games ---")
total_matches = matches.shape[0] // 2
print(f"  Total team-match rows: {matches.shape[0]:,}")
print(f"  Estimated unique matches: {total_matches:,}")

# --- STEP 6: Data Types ---
print("\n" + "=" * 70)
print("DATA TYPES (Are numbers actually numbers?)")
print("=" * 70)

print("\nRound Stats - key columns:")
key_cols_round = ['year', 'round', 'kicks', 'goals', 'disposals', 'fantasy_points', 'score', 'margin']
for col in key_cols_round:
    dtype = round_stats[col].dtype
    print(f"  {col}: {dtype}")

print("\nMatches - key columns:")
key_cols_match = ['year', 'round', 'team_score', 'opponent_score', 'margin']
for col in key_cols_match:
    dtype = matches[col].dtype
    print(f"  {col}: {dtype}")

# --- STEP 7: Outlier Check ---
print("\n" + "=" * 70)
print("OUTLIER CHECK (Extreme values)")
print("=" * 70)

print("\nGoals per game:")
print(f"  Min: {round_stats['goals'].min()}")
print(f"  Max: {round_stats['goals'].max()}")
print(f"  Mean: {round_stats['goals'].mean():.2f}")
print(f"  Std: {round_stats['goals'].std():.2f}")
print(f"  Players with >10 goals in a game: {(round_stats['goals'] > 10).sum():,}")
print(f"  Players with >15 goals in a game: {(round_stats['goals'] > 15).sum():,}")

print("\nDisposals per game:")
print(f"  Min: {round_stats['disposals'].min()}")
print(f"  Max: {round_stats['disposals'].max()}")
print(f"  Mean: {round_stats['disposals'].mean():.2f}")
print(f"  Std: {round_stats['disposals'].std():.2f}")
print(f"  Players with >40 disposals: {(round_stats['disposals'] > 40).sum():,}")
print(f"  Players with >50 disposals: {(round_stats['disposals'] > 50).sum():,}")

# --- STEP 8: Sample — Understand the Grain ---
print("\n" + "=" * 70)
print("SAMPLE: ONE PLAYER'S CAREER (Understand the Grain)")
print("=" * 70)

sample_player_id = round_stats['player_id'].iloc[0]
sample = round_stats[round_stats['player_id'] == sample_player_id].head(5)

print(f"\nPlayer ID: {sample_player_id}")
print(f"Total games for this player: {len(round_stats[round_stats['player_id'] == sample_player_id])}")
print(f"\nFirst 5 games:")
display(sample[['year', 'round', 'team', 'opponent', 'result', 'kicks', 'goals', 'disposals', 'fantasy_points']])

# --- STEP 9: FIXES ---
print("\n" + "=" * 70)
print("APPLYING FIXES")
print("=" * 70)

# FIX 1: Remove duplicates
before_len = {name: len(df) for name, df in [("players_info", players_info), ("round_stats", round_stats), ("seasonal_stats", seasonal_stats), ("matches", matches)]}
players_info = players_info.drop_duplicates()
round_stats = round_stats.drop_duplicates()
seasonal_stats = seasonal_stats.drop_duplicates()
matches = matches.drop_duplicates()
print("\n✅ Fix 1: Duplicates removed")
for name, before, after in [("players_info", before_len["players_info"], len(players_info)), 
                              ("round_stats", before_len["round_stats"], len(round_stats)), 
                              ("seasonal_stats", before_len["seasonal_stats"], len(seasonal_stats)), 
                              ("matches", before_len["matches"], len(matches))]:
    removed = before - after
    if removed > 0:
        print(f"   {name}: {before:,} → {after:,} (-{removed:,})")

# FIX 2: Clean team names
for df in [players_info, round_stats, seasonal_stats, matches]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
teams_after = matches['team_name'].nunique()
print(f"\n✅ Fix 2: Team names cleaned ({teams_after} unique teams)")

# FIX 3: Negative disposals
negative_count = (round_stats['disposals'] < 0).sum()
round_stats.loc[round_stats['disposals'] < 0, 'disposals'] = np.nan
print(f"\n✅ Fix 3: {negative_count} negative disposals set to NaN")

# FIX 4: Check missing players
round_player_ids = set(round_stats['player_id'].unique())
info_player_ids = set(players_info['id'].unique())
missing_from_info = round_player_ids - info_player_ids
print(f"\n✅ Fix 4: {len(missing_from_info)} players in Round Stats but not in Player Info (noted)")

# FIX 5: Missing score column
print(f"\n✅ Fix 5: 'score' column in Round Stats is 100% missing → Use 'team_score' from Matches instead")

# --- FINAL SUMMARY ---
print("\n" + "=" * 70)
print("TASK 1 COMPLETE — CLEAN DATA SUMMARY")
print("=" * 70)

print(f"\n{'File':<20} {'Rows':>10} {'Columns':>10}")
print("-" * 45)
print(f"{'Players Info':<20} {len(players_info):>10,} {players_info.shape[1]:>10}")
print(f"{'Round Stats':<20} {len(round_stats):>10,} {round_stats.shape[1]:>10}")
print(f"{'Seasonal Stats':<20} {len(seasonal_stats):>10,} {seasonal_stats.shape[1]:>10}")
print(f"{'Matches':<20} {len(matches):>10,} {matches.shape[1]:>10}")

print(f"\nCoverage:")
print(f"  Years: {matches['year'].min()} to {matches['year'].max()}")
print(f"  Teams: {matches['team_name'].nunique()}")
print(f"  Players: {round_stats['player_id'].nunique():,}")
print(f"  Matches: {len(matches) // 2:,}")

print(f"\nFixes Applied:")
print(f"  ✅ DtypeWarning fixed (low_memory=False)")
print(f"  ✅ Team names cleaned (whitespace removed)")
print(f"  ✅ Duplicates removed")
print(f"  ✅ Negative disposals set to NaN")
print(f"  ✅ Missing 'score' column noted (use team_score from Matches)")

print(f"\nReady for Task 2!")

All 4 files loaded successfully!
Players Info:       2,848 rows x 16 columns
Round Stats:      274,089 rows x 36 columns
Seasonal Stats:    25,491 rows x 54 columns
Matches:           15,808 rows x 19 columns

FILE 1: PLAYER INFO
Grain: 1 row = 1 player
Columns: ['id', 'player_name', 'player_full_name', 'first_name', 'last_name', 'born_date', 'debut_date', 'debut_age', 'last_date', 'last_age', 'height', 'weight', 'profile_pic', 'player_link', 'player_common_names', 'player_teams']


,id,player_name,born_date,debut_date,height,weight,player_teams
0,43261,Ryan Abbott,1991-06-25,2018-08-02,200,100,"{Geelong Cats,St Kilda Saints}"
1,43262,Gary Ablett,1984-05-14,2002-03-30,182,87,"{Geelong Cats,Gold Coast Suns}"
2,43276,Leek Aleer,2001-08-21,2022-07-30,195,85,NaN



FILE 2: ROUND-BY-ROUND STATS (Biggest file)
Grain: 1 row = 1 player in 1 game
Columns: ['id', 'team', 'year', 'career_game_count', 'opponent', 'round', 'result', 'jersey_num', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assist', 'percentage_of_game_played', 'player_id', 'match_date', 'fantasy_points', 'score', 'margin']


,player_id,team,year,round,opponent,result,kicks,goals,disposals,fantasy_points
0,45552,Hawthorn Hawks,1994,21,Richmond Tigers,W,5.0,NaN,2.0,36
1,44356,Geelong Cats,2024,1,St Kilda Saints,W,5.0,NaN,NaN,23
2,45955,Essendon Bombers,1999,10,Adelaide Crows,W,14.0,NaN,14.0,67



FILE 3: SEASONAL STATS
Grain: 1 row = 1 player in 1 season
Columns: ['player_id', 'year', 'team', 'is_finals', 'games_played', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assists', 'total_score', 'total_fantasy_points', 'total_percentage_played', 'avg_kicks', 'avg_marks', 'avg_handballs', 'avg_disposals', 'avg_goals', 'avg_behinds', 'avg_hit_outs', 'avg_tackles', 'avg_rebound_50s', 'avg_inside_50s', 'avg_clearances', 'avg_clangers', 'avg_free_kicks_for', 'avg_free_kicks_against', 'avg_contested_possessions', 'avg_uncontested_possessions', 'avg_contested_marks', 'avg_marks_inside_50', 'avg_one_percenters', 'avg_bounces', 'avg_goal_assists', 'avg_score', 'avg_fantasy_points', 'avg_percentage_played']


...


FILE 4: MATCH RESULTS
Grain: 1 row = 1 team in 1 match
Columns: ['id', 'team_name', 'round', 'match_date', 'year', 'home_away', 'opponent', 'team_quarter_scores', 'team_score', 'opponent_quarter_scores', 'opponent_score', 'result', 'margin', 'venue', 'crowd', 'team_goals_kicked', 'team_behinds', 'opponent_goals_kicked', 'opponent_behinds']


,team_name,year,round,match_date,home_away,opponent,team_score,opponent_score,result,margin,venue
0,Hawthorn Hawks,1994,QF,1994-09-10,A,North Melbourne Kangaroos,91,114,L,23,Waverley Park
1,North Melbourne Kangaroos,1994,QF,1994-09-10,H,Hawthorn Hawks,114,91,W,6,Waverley Park
2,North Melbourne Kangaroos,2008,10,2008-05-31,A,Brisbane Lions,98,129,L,-31,The Gabba



DATA QUALITY CHECK: MISSING VALUES
(NaN = Not a Number = empty cell = missing data)

--- Players Info (2,848 rows) ---
  ⚠️  5,078 missing values (11.1% of all cells)
     - player_common_names: 2,773 missing (97.4%)
     - profile_pic: 2,211 missing (77.6%)
     - player_teams: 94 missing (3.3%)

--- Round Stats (274,089 rows) ---
  ⚠️  1,608,029 missing values (16.3% of all cells)
     - score: 274,089 missing (100.0%)
     - brownlow_votes: 110,481 missing (40.3%)
     - goal_assist: 103,600 missing (37.8%)
     - bounces: 97,343 missing (35.5%)
     - hit_outs: 96,629 missing (35.3%)
     - marks_inside_50: 91,378 missing (33.3%)
     - contested_marks: 90,677 missing (33.1%)
     - behinds: 81,093 missing (29.6%)
     - goals: 74,737 missing (27.3%)
     - percentage_of_game_played: 69,682 missing (25.4%)
     - clearances: 63,919 missing (23.3%)
     - rebound_50s: 62,856 missing (22.9%)
     - one_percenters: 60,146 missing (21.9%)
     - free_kicks_for: 53,842 missing (19.6%)


,year,round,team,opponent,result,kicks,goals,disposals,fantasy_points
0,1994,21,Hawthorn Hawks,Richmond Tigers,W,5.0,NaN,2.0,36
25,1995,9,Hawthorn Hawks,Adelaide Crows,L,14.0,NaN,13.0,71
39,2000,16,Hawthorn Hawks,Collingwood Magpies,W,12.0,NaN,18.0,92
62,2001,16,Hawthorn Hawks,Collingwood Magpies,W,7.0,NaN,7.0,46
68,1996,1,Hawthorn Hawks,Fitzroy Lions,W,14.0,1.0,14.0,82



APPLYING FIXES

✅ Fix 1: Duplicates removed
   players_info: 2,848 → 2,843 (-5)
   round_stats: 274,089 → 274,079 (-10)
   seasonal_stats: 25,491 → 25,481 (-10)

✅ Fix 2: Team names cleaned (20 unique teams)

✅ Fix 3: 722 negative disposals set to NaN

✅ Fix 4: 266 players in Round Stats but not in Player Info (noted)

✅ Fix 5: 'score' column in Round Stats is 100% missing → Use 'team_score' from Matches instead

TASK 1 COMPLETE — CLEAN DATA SUMMARY

File                       Rows    Columns
---------------------------------------------
Players Info              2,843         16
Round Stats             274,079         36
Seasonal Stats           25,481         54
Matches                  15,808         19

Coverage:
  Years: 1983 to 2025
  Teams: 20
  Players: 3,109
  Matches: 7,904

Fixes Applied:
  ✅ DtypeWarning fixed (low_memory=False)
  ✅ Team names cleaned (whitespace removed)
  ✅ Duplicates removed
  ✅ Negative disposals set to NaN
  ✅ Missing 'score' column noted (use team_sc

In [3]:
# =============================================================================
# TASK 2: DEFINE PREDICTION TARGETS
# =============================================================================
# What we're doing: Define exactly what we want to predict
# Why: Without clear targets, models can't be built or evaluated
# =============================================================================

import pandas as pd
import numpy as np
import os

# --- Reload clean data from Task 1 ---
data_path = r"C:\Internship\Netixsol\week-3\day-1\data\afl_datasets"

players_info = pd.read_csv(os.path.join(data_path, "afl_players_info_raw.csv"))
round_stats = pd.read_csv(os.path.join(data_path, "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"))
seasonal_stats = pd.read_csv(os.path.join(data_path, "afl_players_seasonal_stats_raw.csv"), low_memory=False)
matches = pd.read_csv(os.path.join(data_path, "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv"))

# Apply fixes from Task 1
for df in [players_info, round_stats, seasonal_stats, matches]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
players_info = players_info.drop_duplicates()
round_stats = round_stats.drop_duplicates()
seasonal_stats = seasonal_stats.drop_duplicates()
matches = matches.drop_duplicates()
round_stats.loc[round_stats['disposals'] < 0, 'disposals'] = np.nan

print("Data loaded and cleaned!")

# =============================================================================
# TARGET 1: MATCH WINNER (Classification)
# =============================================================================
print("\n" + "=" * 70)
print("TARGET 1: MATCH WINNER")
print("=" * 70)

# What we're defining: For each match, who won?
# Level: Match level (one row per match)

# First, let's understand the data
print("\nUnderstanding the match data:")
print(f"  Total team-match rows: {len(matches):,}")
print(f"  Unique matches: {len(matches) // 2:,}")
print(f"  Result values: {matches['result'].unique()}")
print(f"  Result counts:")
print(matches['result'].value_counts())

# Create match-level target
# Each match appears twice (once per team), so we take one row per match
# We'll use the home team's perspective

# Filter to home team only
home_matches = matches[matches['home_away'] == 'H'].copy()

# Define target: home_win (1) or away_win (0)
# If home team wins (result='W'), home_win = 1
# If home team loses (result='L'), home_win = 0
# If draw (result='D'), we'll exclude from classification
home_matches['home_win'] = (home_matches['result'] == 'W').astype(int)
home_matches['is_draw'] = (home_matches['result'] == 'D').astype(int)

# Also keep margin as regression target
# Margin = team_score - opponent_score (positive = home win)
home_matches['margin'] = home_matches['team_score'] - home_matches['opponent_score']

print("\n--- Match Winner Target Created ---")
print(f"  Level: Match (one row per match)")
print(f"  Total matches: {len(home_matches):,}")
print(f"  Home wins: {(home_matches['home_win'] == 1).sum():,} ({(home_matches['home_win'] == 1).mean()*100:.1f}%)")
print(f"  Away wins: {(home_matches['home_win'] == 0).sum():,} ({(home_matches['home_win'] == 0).mean()*100:.1f}%)")
print(f"  Draws: {(home_matches['is_draw'] == 1).sum():,} ({(home_matches['is_draw'] == 1).mean()*100:.1f}%)")
print(f"  Margin range: {home_matches['margin'].min()} to {home_matches['margin'].max()}")
print(f"  Margin mean: {home_matches['margin'].mean():.2f}")

print("\nSample of match winner target:")
display(home_matches[['team_name', 'year', 'round', 'match_date', 'opponent', 'team_score', 'opponent_score', 'result', 'home_win', 'margin']].head(10))

# =============================================================================
# TARGET 2: TOP PLAYER (Multiple Versions)
# =============================================================================
print("\n" + "=" * 70)
print("TARGET 2: TOP PLAYER")
print("=" * 70)

# What we're defining: For each match, who was the top player?
# Level: Player-game level (one row per player per game)

# First, let's understand the data
print("\nUnderstanding the player-game data:")
print(f"  Total player-game rows: {len(round_stats):,}")
print(f"  Unique players: {round_stats['player_id'].nunique():,}")
print(f"  Unique years: {round_stats['year'].nunique()}")

# Create player-game targets
player_game = round_stats.copy()

# Version 1: Top Disposal-Getter
# Disposals = kicks + handballs
player_game['disposals_calc'] = player_game['kicks'] + player_game['handballs']
# Use existing disposals column if available, otherwise use calculated
player_game['disposals_target'] = player_game['disposals'].fillna(player_game['disposals_calc'])

# Version 2: Top Goal-Kicker
# Goals direct from data
player_game['goals_target'] = player_game['goals']

# Version 3: Fantasy Points
# Already in data
player_game['fantasy_target'] = player_game['fantasy_points']

# Version 4: Brownlow-Style Score (custom weighted composite)
# Formula: contested_poss*2 + clearances*3 + tackles*2 + inside_50s*2 + rebound_50s*2 + goals*6 + behinds*1
player_game['brownlow_target'] = (
    player_game['contested_possessions'].fillna(0) * 2 +
    player_game['clearances'].fillna(0) * 3 +
    player_game['tackles'].fillna(0) * 2 +
    player_game['inside_50s'].fillna(0) * 2 +
    player_game['rebound_50s'].fillna(0) * 2 +
    player_game['goals'].fillna(0) * 6 +
    player_game['behinds'].fillna(0) * 1
)

print("\n--- Top Player Targets Created ---")
print(f"  Level: Player-game (one row per player per game)")
print(f"  Total player-game rows: {len(player_game):,}")

# Show top performers for each target
print("\n--- Top 10 Disposal-Getters (All Time) ---")
top_disposals = player_game.nlargest(10, 'disposals_target')[['player_id', 'team', 'year', 'round', 'opponent', 'disposals_target']]
display(top_disposals)

print("\n--- Top 10 Goal-Kickers (All Time) ---")
top_goals = player_game.nlargest(10, 'goals_target')[['player_id', 'team', 'year', 'round', 'opponent', 'goals_target']]
display(top_goals)

print("\n--- Top 10 Fantasy Points (All Time) ---")
top_fantasy = player_game.nlargest(10, 'fantasy_target')[['player_id', 'team', 'year', 'round', 'opponent', 'fantasy_target']]
display(top_fantasy)

print("\n--- Top 10 Brownlow-Style Score (All Time) ---")
top_brownlow = player_game.nlargest(10, 'brownlow_target')[['player_id', 'team', 'year', 'round', 'opponent', 'brownlow_target']]
display(top_brownlow)

# =============================================================================
# DATA DICTIONARY
# =============================================================================
print("\n" + "=" * 70)
print("DATA DICTIONARY")
print("=" * 70)

dictionary = """
┌─────────────────────┬─────────────────────────────────────────────┬─────────────────────────────────────────────┬──────────────────┐
│ Target Name         │ Definition                                  │ Formula                                     │ Aggregation Level│
├─────────────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼──────────────────┤
│ home_win            │ Did home team win?                          │ 1 if result='W', 0 if result='L'           │ Match            │
│ margin              │ Score difference                            │ team_score - opponent_score                 │ Match            │
│ disposals_target    │ Player's total disposals in game            │ kicks + handballs                           │ Player-Game      │
│ goals_target        │ Player's goals in game                      │ goals (direct from data)                    │ Player-Game      │
│ fantasy_target      │ AFL Fantasy Points score                    │ fantasy_points (direct from data)           │ Player-Game      │
│ brownlow_target     │ Weighted composite score                    │ contested_poss*2 + clearances*3 + ...       │ Player-Game      │
└─────────────────────┴─────────────────────────────────────────────┴─────────────────────────────────────────────┴──────────────────┘
"""
print(dictionary)

# =============================================================================
# JUSTIFICATION FOR CHOICES
# =============================================================================
print("=" * 70)
print("JUSTIFICATION FOR CHOICES")
print("=" * 70)

justification = """
WHY CLASSIFICATION + REGRESSION FOR MATCH WINNER?
- Classification (home_win): Simple, interpretable, good for betting decisions
- Regression (margin): More informative, captures confidence level
- Using both gives complementary information

WHY MULTIPLE TOP PLAYER TARGETS?
- Disposals: Measures overall involvement (midfielders excel)
- Goals: Measures scoring impact (forwards excel)
- Fantasy Points: Comprehensive measure (rewards all contributions)
- Brownlow Score: Emphasizes contested ball and impact (what umpires value)

WHY EXCLUDE DRAWS FROM CLASSIFICATION?
- Draws are rare (<3% of matches)
- Including them confuses the model (3 classes vs 2)
- Better to handle draws separately if needed

WHY USE PLAYER-GAME LEVEL FOR TOP PLAYER?
- "Top player" is per-game, not per-season
- We want to predict who will be best in UPCOMING game
- Season averages don't capture single-game dominance
"""
print(justification)

# =============================================================================
# SUMMARY
# =============================================================================
print("=" * 70)
print("TASK 2 COMPLETE — PREDICTION TARGETS DEFINED")
print("=" * 70)

print(f"""
WHAT WE DEFINED:
1. Match Winner (Classification): home_win = {1 if home_matches['home_win'].mean() > 0.5 else 0} (home team wins {home_matches['home_win'].mean()*100:.1f}% of time)
2. Match Margin (Regression): Average margin = {home_matches['margin'].mean():.1f} points
3. Top Disposals: Average = {player_game['disposals_target'].mean():.1f} per game
4. Top Goals: Average = {player_game['goals_target'].mean():.1f} per game
5. Top Fantasy: Average = {player_game['fantasy_target'].mean():.1f} per game
6. Top Brownlow: Average = {player_game['brownlow_target'].mean():.1f} per game

READY FOR TASK 3: We now know exactly what we're predicting.
""")

Data loaded and cleaned!

TARGET 1: MATCH WINNER

Understanding the match data:
  Total team-match rows: 15,808
  Unique matches: 7,904
  Result values: ['L' 'W' 'D']
  Result counts:
result
L    7839
W    7839
D     130
Name: count, dtype: int64

--- Match Winner Target Created ---
  Level: Match (one row per match)
  Total matches: 7,904
  Home wins: 4,669 (59.1%)
  Away wins: 3,235 (40.9%)
  Draws: 65 (0.8%)
  Margin range: -164 to 186
  Margin mean: 8.97

Sample of match winner target:


,team_name,year,round,match_date,opponent,team_score,opponent_score,result,home_win,margin
1,North Melbourne Kangaroos,1994,QF,1994-09-10,Hawthorn Hawks,114,91,W,1,23
5,Sydney Swans,2019,12,2019-06-09,West Coast Eagles,116,71,W,1,45
8,W. Bulldogs,2000,9,2000-05-05,St Kilda Saints,105,104,W,1,1
11,W. Bulldogs,2000,12,2000-05-28,Melbourne Demons,108,83,W,1,25
13,W. Bulldogs,2000,15,2000-06-18,North Melbourne Kangaroos,106,102,W,1,4
16,W. Bulldogs,2000,20,2000-07-22,Collingwood Magpies,100,91,W,1,9
18,W. Bulldogs,2000,22,2000-08-04,Hawthorn Hawks,66,81,L,0,-15
20,W. Bulldogs,2001,1,2001-03-31,St Kilda Saints,107,112,L,0,-5
23,W. Bulldogs,2001,17,2001-07-27,Richmond Tigers,100,102,L,0,-2
27,W. Bulldogs,2001,22,2001-09-02,Melbourne Demons,123,133,L,0,-10



TARGET 2: TOP PLAYER

Understanding the player-game data:
  Total player-game rows: 274,079
  Unique players: 3,109
  Unique years: 43

--- Top Player Targets Created ---
  Level: Player-game (one row per player per game)
  Total player-game rows: 274,079

--- Top 10 Disposal-Getters (All Time) ---


,player_id,team,year,round,opponent,disposals_target
167386,44910,North Melbourne Kangaroos,2025,23,Richmond Tigers,54.0
213046,44510,Hawthorn Hawks,2018,1,Collingwood Magpies,54.0
269304,43262,Gold Coast Suns,2012,10,Collingwood Magpies,53.0
165386,44585,Brisbane Lions,2019,23,Richmond Tigers,51.0
270187,45089,Adelaide Crows,2011,22,Gold Coast Suns,51.0
245586,44510,Hawthorn Hawks,2017,9,Collingwood Magpies,50.0
261052,44510,Hawthorn Hawks,2018,15,Greater Western Sydney Giants,50.0
93097,45048,Collingwood Magpies,2012,17,Hawthorn Hawks,49.0
93542,43262,Gold Coast Suns,2013,17,Collingwood Magpies,49.0
181913,43669,Western Bulldogs,2025,10,Essendon Bombers,49.0



--- Top 10 Goal-Kickers (All Time) ---


...


--- Top 10 Fantasy Points (All Time) ---


,player_id,team,year,round,opponent,fantasy_target
23628,45408,North Melbourne Kangaroos,1996,17,Melbourne Demons,210
58117,45679,St Kilda Saints,1992,13,Sydney Swans,209
129212,43847,Hawthorn Hawks,2012,10,North Melbourne Kangaroos,204
120002,45684,North Melbourne Kangaroos,1990,2,Richmond Tigers,201
164556,44786,St Kilda Saints,2016,23,Brisbane Lions,200
170671,45677,Essendon Bombers,1999,3,Sydney Swans,198
261052,44510,Hawthorn Hawks,2018,15,Greater Western Sydney Giants,195
64082,43884,Carlton Blues,2017,13,Gold Coast Suns,194
109698,45679,Sydney Swans,1995,19,Fitzroy Lions,194
210257,45001,Essendon Bombers,2012,6,Brisbane Lions,193



--- Top 10 Brownlow-Style Score (All Time) ---


,player_id,team,year,round,opponent,brownlow_target
214742,43262,Gold Coast Suns,2017,6,North Melbourne Kangaroos,163.0
171900,44630,Melbourne Demons,2021,10,Adelaide Crows,155.0
193461,44281,Western Bulldogs,2024,4,Geelong Cats,149.0
234226,43262,Gold Coast Suns,2011,8,Adelaide Crows,146.0
117604,43642,Carlton Blues,2019,19,Adelaide Crows,145.0
175694,43262,Gold Coast Suns,2012,3,Essendon Bombers,144.0
197587,44194,West Coast Eagles,2006,5,Brisbane Lions,143.0
213659,44332,Gold Coast Suns,2018,2,Carlton Blues,140.0
211523,45046,North Melbourne Kangaroos,2011,11,Adelaide Crows,139.0
42888,44649,Essendon Bombers,2021,16,Geelong Cats,137.0



DATA DICTIONARY

┌─────────────────────┬─────────────────────────────────────────────┬─────────────────────────────────────────────┬──────────────────┐
│ Target Name         │ Definition                                  │ Formula                                     │ Aggregation Level│
├─────────────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼──────────────────┤
│ home_win            │ Did home team win?                          │ 1 if result='W', 0 if result='L'           │ Match            │
│ margin              │ Score difference                            │ team_score - opponent_score                 │ Match            │
│ disposals_target    │ Player's total disposals in game            │ kicks + handballs                           │ Player-Game      │
│ goals_target        │ Player's goals in game                      │ goals (direct from data)                    │ Player-Game      │
│ fantasy_target      │ AFL Fantasy Po